# Implementing a Basic Vector Search

In [16]:
from dotenv import load_dotenv

load_dotenv()

True

## Step 1: Install the Qdrant Client

In [18]:
# !pip install qdrant-client

## Step 2: Import Required Libraries

In [19]:
from qdrant_client import QdrantClient, models

## Step 3: Connect to Qdrant Cloud

In [20]:
client = QdrantClient(url=os.getenv("QDRANT_URL"))

## Step 4: Create a Collection

In [21]:
# Define the collection name
collection_name = "my_first_collection"

# Create the collection with specified vector parameters
client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=4,  # Dimensionality of the vectors
        distance=models.Distance.COSINE  # Distance metric for similarity search
    )
)

True

## Step 5: Verify Collection Creation

In [22]:
# Retrieve and display the list of collections
collections = client.get_collections()
print("Existing collections:", collections)

Existing collections: collections=[CollectionDescription(name='midjourney'), CollectionDescription(name='my_first_collection')]


## Step 6: Insert Points into the Collection

In [23]:
# Define the vectors to be inserted
points = [
    models.PointStruct(
        id=1,
        vector=[0.1, 0.2, 0.3, 0.4],  # 4D vector
        payload={"category": "example"}  # Metadata (optional)
    ),
    models.PointStruct(
        id=2,
        vector=[0.2, 0.3, 0.4, 0.5],
        payload={"category": "demo"}
    )
]

# Insert vectors into the collection
client.upsert(
    collection_name=collection_name,
    points=points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

## Step 7: Retrieve Collection Details

In [24]:
collection_info = client.get_collection(collection_name)
print("Collection info:", collection_info)

Collection info: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=2 segments_count=5 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=4, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None), wal_c

## Step 8: Run Your First Similarity Search

In [25]:
query_vector = [0.08, 0.14, 0.33, 0.28]

search_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=1  # Return the top 1 most similar vector
)

print("Search results:", search_results)

Search results: points=[ScoredPoint(id=1, version=1, score=0.97642946, payload={'category': 'example'}, vector=None, shard_key=None, order_value=None)]


In [32]:
# Define a filter by caterogy
filter_by_category = models.Filter(
    must=models.FieldCondition(
        key="category",
        match=models.MatchValue(value="example")
    )
)

filtered_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    query_filter=filter_by_category, # Use filter to refine the search 
    limit=1  # Return the top 1 most similar vector
)

print("Filtered results:")
filtered_results

Filtered results:


QueryResponse(points=[ScoredPoint(id=1, version=1, score=0.97642946, payload={'category': 'example'}, vector=None, shard_key=None, order_value=None)])